In [0]:
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"])
catalog_name = dbutils.widgets.get("catalog")

dbutils.widgets.text("bronze_table", "mens_volleyball_raw")
dbutils.widgets.text("silver_table", "mens_volleyball_clean")

bronze_table = dbutils.widgets.get("bronze_table")
silver_table = dbutils.widgets.get("silver_table")

bronze_table_name = f"{catalog_name}.trezio2005_bronze.{bronze_table}"
silver_table_name = f"{catalog_name}.trezio2005_silver.{silver_table}"

In [0]:
df_bronze = spark.table(bronze_table_name)
display(df_bronze)

Data cleaning pipeline for the transition from bronze to silver

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

def clean_data(df):
    #dropping duplicates, adding metadata, creating key for merge
    df = df.withColumn("Match_ID", F.md5(F.concat_ws("-", F.col("Date"), F.col("Team_1"), F.col("Team_2"))))
    df = df.dropDuplicates(["Match_ID"])
    df = df.withColumn("_ingested_at", F.current_timestamp())
    df = df.withColumn("Date", F.to_timestamp(F.col("Date"), "dd.MM.yyyy, HH:mm"))

    #fixing types from string to numeric
    percentage_columns = [
        "T1_Rec_Pos", "T1_Rec_Perf", "T1_Att_Kill_Perc", "T1_Att_Eff", "T1_Srv_Eff",
        "T2_Rec_Pos", "T2_Rec_Perf", "T2_Att_Kill_Perc", "T2_Att_Eff", "T2_Srv_Eff"
    ]   

    for col in percentage_columns:
        clean_string = F.trim(F.regexp_replace(F.regexp_replace(F.col(col), "%", ""), ",", "."))
        df = df.withColumn(col, clean_string.cast(DoubleType()) / 100.0)
        
    string_to_int_cols = ["T1_Srv_Err", "T1_Att_Sum", "T1_Blk_As", "T2_Srv_Err", "T2_Att_Sum", "T2_Blk_As"]

    for col in string_to_int_cols:
        clean_val = F.trim(F.regexp_replace(F.regexp_replace(F.col(col), "%", ""), ",", "."))
        df = df.withColumn(col, clean_val.cast(DoubleType()).cast(IntegerType()))

    double_to_int_cols = [
        "T1_Sum", "T1_BP", "T1_Srv_Sum", "T1_Srv_Ace", "T1_Rec_Sum", 
        "T1_Rec_Err", "T1_Att_Err", "T1_Att_Blk", "T1_Att_Kill", "T1_Blk_Sum"
    ]
    for col in double_to_int_cols:
        df = df.withColumn(col, F.col(col).cast(IntegerType()))

    #filling null values
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (IntegerType, DoubleType))]

    df = df.fillna("Unknown", subset=string_cols)
    df = df.fillna(0, subset=numeric_cols)

    #adding column for better analysis
    df = df.withColumn(
        "Winner_Team_Name",
        F.when(F.col("Winner") == 1, F.col("Team_1"))
         .when(F.col("Winner") == 2, F.col("Team_2"))
         .otherwise("Unknown")
    )
    
    return df

SCD TYPE 1 - old data is overwritten by the new data (upsert)

SCD TYPE 2 - saves historical data, uses columns like active, valid_from, valid_to to keep all versions of the data

In my case type 1 has more sense 

In [0]:
from delta.tables import DeltaTable

df_clean = clean_data(df_bronze)
if spark.catalog.tableExists(silver_table_name):
    silver_table = DeltaTable.forName(spark, silver_table_name)
    #Merge, SCD Type 1
    (silver_table.alias("target").merge(
            source=df_clean.alias("source"),
            condition="target.Match_ID = source.Match_ID"
        )
        .whenMatchedUpdateAll() #adding all updated columns (update)
        .whenNotMatchedInsertAll() #adding all new columns (insert)
        .execute()
    )
else:
    df_clean.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)